In [33]:
# A. PHẦN PHỐI CẢNH

#CÁC BƯỚC:

# 1. đọc ảnh 
# 2. Nhị phân hóa.
# 3. Phát hiện cạnh (lề) giấy A4.
# 4. phối lại cảnh ( nhờ 4 góc giấy)
# 5. phối lại các nút định vị lớn 1 lần nửa

In [34]:
# 1.ĐỌC ẢNH. =======================================================================================================
import cv2 
import numpy as np
import imutils  # thư viện tích
 
img = cv2.imread("chup8_nguoc.jpg")
 
original_img = img.copy()

ratio = img.shape[0] / 750.0  #tính tỉ lệ khi resize xuông còn 750 height là bao nhiêu(để dùng cho phần sau)

fixed_img = imutils.resize(img, height=750) 




In [35]:
# 2. Nhị phân hóa.===================================================================================================

gray = cv2.cvtColor(fixed_img, cv2.COLOR_BGR2GRAY)  

gauss = cv2.GaussianBlur(gray, (3,3), 0)

edged = cv2.Canny(gauss, 50, 150)



In [36]:
# 3. phát hiện lề.=====================================================================================================

# Tìm các đường bao (contours) của ảnh và làm nổi bật đường bao có diện tích lớn nhất.

cnts = cv2.findContours(edged.copy(), cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
cnts = imutils.grab_contours(cnts)
cnts = sorted(cnts, key=cv2.contourArea, reverse=True)[:5]

#duyệt qua mỗi đường biên lấy được
for c in cnts:
    
    # contour approximation
    peri = cv2.arcLength(c, True) 
    approx = cv2.approxPolyDP(c, 0.02 * peri, True) # xấp sỉ đa giác 


    # nếu có xấp xỉ(aprrox) khoảng 4 điểm(point)
    if len(approx) == 4:    
        # thì cho rằng đó là đường viền của tờ A4
        screenCnt = approx
        break




In [37]:
# 4. chuẩn hóa góc và phối lại cảnh=====================================================================================


def points_normalization(pts):   #hàm chuẩn hóa các góc tài liệu và sắp xếp lại thứ tự 4 điểm

    # pts là mảng gồm 4 điểm (4x2)

    # trả về mảng 4 điểm theo thứ tự: [trai_tren, phai_tren, phai_duoi, trai_duoi]
    rect = np.zeros((4, 2), dtype="float32")
    

    # tính tổng và hiệu của các tọa độ x, y
    s = pts.sum(axis=1)
    # hiệu (x - y) cho mỗi điểm
    diff = np.diff(pts, axis=1)

    # xác định vị trí các góc
    rect[0] = pts[np.argmin(s)]        # top-left có tổng nhỏ nhất
    rect[2] = pts[np.argmax(s)]        # bottom-right có tổng lớn nhất

    rect[1] = pts[np.argmin(diff)]     # top-right có hiệu nhỏ nhất
    rect[3] = pts[np.argmax(diff)]     # bottom-left có hiệu lớn nhất

    return rect


# mảng các điểm 4 góc 
pst = screenCnt.reshape(4, 2)

# dùng hàm chuẩn hóa ở trên và nhân với tỉ lệ ảnh gốc để đúng tỉ lệ ảnh ban đầu
pst = points_normalization(pst * ratio)
(tl, tr, br, bl) = pst

# Tính chiều rộng của ảnh mới, bằng khoảng cách lớn nhất(max) giữa các tọa độ x của các điểm cực trị của hình chữ nhật.

widthA = np.sqrt(((br[0] - bl[0]) ** 2) + ((br[1] - bl[1]) ** 2))
widthB = np.sqrt(((tr[0] - tl[0]) ** 2) + ((tr[1] - tl[1]) ** 2))
maxWidth = max(int(widthA), int(widthB))

# Tính chiều cao của ảnh mới, bằng khoảng cách lớn nhất(max) giữa các tọa độ y của các điểm cực trị của hình chữ nhật.
heightA = np.sqrt(((tr[0] - br[0]) ** 2) + ((tr[1] - br[1]) ** 2))
heightB = np.sqrt(((tl[0] - bl[0]) ** 2) + ((tl[1] - bl[1]) ** 2))
maxHeight = max(int(heightA), int(heightB))

# Hình thành (tạo ra) các tọa độ cuối cùng của các điểm tham chiếu.
dst = np.array([
    [0, 0],                       # top - left
    [maxWidth - 1, 0],             # top - right 

    [maxWidth - 1, maxHeight - 1], # bottom - right 
    [0, maxHeight - 1]             # bottm - left 
], dtype="float32")

# Xác định ma trận để thực hiện phép biến đổi phối cảnh.(perspective transformation)
M = cv2.getPerspectiveTransform(pst, dst)

# phép biến đổi phối cảnh.(perspective transformation)
warped = cv2.warpPerspective(original_img, M, (maxWidth, maxHeight))

warped = imutils.resize(warped, height=750) ##thu nhỏ ảnh để hiển thị vừa màn hình


# kết quả của phép biến đổi phối cảnh.(perspective transformation)

# cv2.imshow("anh da phoi canh", edged) 
# cv2.waitKey(0)
# cv2.destroyAllWindows()

###=========================================THÀNH CÔNG PHỐI CẢNH LỀ GIẤY A4


In [38]:
#++++++++++++++++++++++++++++++-------------------------------------------------------------------------------------------------------

In [39]:
# 5. Phối cảnh 1 lần nửa các nút định vị-------------------------------------

gray = cv2.cvtColor(warped, cv2.COLOR_BGR2GRAY)  

gauss = cv2.GaussianBlur(gray, (3,3), 0)

thresh = cv2.Canny(gauss, 75, 200)
 
# thresh = cv2.adaptiveThreshold(gauss, 255,
#                                cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
#                                cv2.THRESH_BINARY_INV, 19, 3)  #nhị phân hóa.

cnts = cv2.findContours(thresh, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
cnts = imutils.grab_contours(cnts)
# cnts = sorted(cnts, key=cv2.contourArea, reverse=True)[:8]


# Lưu các mốc định vị có diện tích lớn
markers = []

for c in cnts:
    
    area = cv2.contourArea(c)

    if area < 200 and area > 150:                 # Ngưỡng để lọc ra 4 ô vuông lớn
        peri = cv2.arcLength(c, True)
        approx = cv2.approxPolyDP(c, 0.02 * peri, True)

        if len(approx) == 4:        # Là hình vuông / hình chữ nhật
            markers.append(approx)
          

centers = [] # tính tâm của các contours
for m in markers:
    M = cv2.moments(m)
    cx = int(M["m10"] / M["m00"])
    cy = int(M["m01"] / M["m00"])
    centers.append((cx, cy))

print(centers)

output = warped.copy()

# Vẽ tâm 4 mốc
for (cx, cy) in centers:
    cv2.circle(output, (cx, cy), 5, (0, 255, 255), -1)  # -1 = vẽ hình tròn đầy


# # Vẽ contour 4 mốc
# cv2.drawContours(output, markers, -1, (0, 255, 255), 2) 



# cv2.imshow("anh da phoi canh", warped)
# # cv2.waitKey(0)

cv2.imshow("nhi phan", thresh)
# cv2.waitKey(0)

cv2.imshow("ve contours", output)
# cv2.waitKey(0)
# cv2.destroyAllWindows()

# print(markers)

[(447, 647), (72, 647), (72, 371), (447, 368), (74, 94), (449, 89)]


In [40]:
# 1. Chuyển list centers thành numpy array float32
pts_6 = np.array(centers, dtype="float32")


# 3. Lấy đúng 4 góc chuẩn

four_corners = points_normalization(pts_6) # dùng lại hàm đã xây dưng ở trên để sắp 4 góc

# four_corners = get_4_corners(pts_6)
(tl, tr, br, bl) = four_corners

print("4 góc đã lọc:", four_corners)

# 4. Tính toán chiều rộng/cao mới (giống hệt bước 4 cũ)
widthA = np.sqrt(((br[0] - bl[0]) ** 2) + ((br[1] - bl[1]) ** 2))
widthB = np.sqrt(((tr[0] - tl[0]) ** 2) + ((tr[1] - tl[1]) ** 2))
maxWidth = max(int(widthA), int(widthB))

heightA = np.sqrt(((tr[0] - br[0]) ** 2) + ((tr[1] - br[1]) ** 2))
heightB = np.sqrt(((tl[0] - bl[0]) ** 2) + ((tl[1] - bl[1]) ** 2))
maxHeight = max(int(heightA), int(heightB))

# 5. Tạo điểm đích (Destination points)
dst = np.array([
    [0, 0],
    [maxWidth - 1, 0],
    [maxWidth - 1, maxHeight - 1],
    [0, maxHeight - 1]
], dtype="float32")

# 6. Thực hiện phối cảnh lần 2 (Fine-tuning)
M_final = cv2.getPerspectiveTransform(four_corners, dst)

warped_final = cv2.warpPerspective(warped, M_final, (maxWidth, maxHeight))

# Hiển thị kết quả cuối cùng
cv2.imshow("Phieu Trac Nghiem Hoan Chinh", warped_final)
cv2.waitKey(0)
cv2.destroyAllWindows()

4 góc đã lọc: [[ 74.  94.]
 [449.  89.]
 [447. 647.]
 [ 72. 647.]]
